# Pipeline from a 1D spectrum

Simulate a BSPEC rate map from a BT-Settl template, build a `RampModel`, sync BSPEC calibrations from KOA if needed, then run `Stage0Pipeline` → `Stage1Pipeline` → `Stage2Pipeline` with defaults and show the stellar-RV preview.

**Environment variables** (set these in your shell / notebook kernel before running):

| Variable | Required when | Purpose |
|---|---|---|
| `KOA_CALIBRATION_CACHE` | Always | Local KOA calibration cache directory (and SQLite DB) |
| `KECK_OBSERVER_EMAIL` | Syncing / downloading from KOA | Keck Observer account email |
| `KECK_OBSERVER_PASSWORD` | Syncing / downloading from KOA | Keck Observer account password |

If `KOA_CALIBRATION_CACHE` is unset, the setup cell sets it under `OUT`.  
The KOA cell below syncs metadata from remote KOA and downloads any **BSPEC** calibration files that are not already in the cache.


In [ ]:
import os

# OUTPUT DIRECTORY — update this to your desired output directory
OUT = "/path/to/hispec_pipeline_example"
os.makedirs(OUT, exist_ok=True)

os.environ.setdefault(
    "KOA_CALIBRATION_CACHE",
    os.path.join(OUT, "cals", "KOA_CALIBRATION_CACHE"),
)
print("output:", OUT)
print("KOA_CALIBRATION_CACHE:", os.environ["KOA_CALIBRATION_CACHE"])


## 1. Sync / verify KOA caldb (BSPEC)


In [ ]:
from hispecdrp.calibrations import verify_koa_calibrations

# Requires env vars: KOA_CALIBRATION_CACHE, KECK_OBSERVER_EMAIL, KECK_OBSERVER_PASSWORD
SPECTROGRAPH = "BSPEC"
summary = verify_koa_calibrations(SPECTROGRAPH)

print(
    f"{summary['spectrograph']} caldb: {summary['n_records']} record(s); "
    f"downloaded {summary['n_downloaded']}/{summary['n_missing']} missing file(s)"
)
print("types:", summary["cal_types"])
if summary["downloaded"]:
    print("downloaded:", summary["downloaded"])
if summary["failed"]:
    print("failed:", summary["failed"])


## 2. Simulated rate map from a 1D spectrum → `RampModel`

Load a BT-Settl template, create a rate map with `SimulateRateMap`, then build a raw UTR `RampModel` and provide the caldb files that were used to create the physical ramp.


In [ ]:
from hispecdrp.simulations.simulate_up_the_ramp import create_ramp_from_spectrum

FIBERS = [1, 2, 3]  # SPECTROGRAPH is set in the KOA cell above

# Build a rate map from a 1D BT-Settl template and a raw UTR RampModel (KOA masters from cache).
ramp, cals, rate_map = create_ramp_from_spectrum(
    OUT,
    spectrograph=SPECTROGRAPH,
    fibers=FIBERS,
    template_filename="lte049.0-2.5-0.0a+0.0.BT-Settl.spec.7.h5",
    mag_ab=3.0,  # brightest non-saturating for these params (~78% full well; J=2.5 saturates)
    filter_band="J",  # BSPEC (yJ)
)

# Data models expose class name, filename, and arrays as attributes:
print(f"ramp type: {ramp.meta.model_type}")
print(f"ramp filename: {ramp.meta.filename}")
print(f"ramp data shape: {ramp.data.shape}")
print(f"Stage-0 cals: {sorted(cals)}")
print(f"rate_map shape: {rate_map.shape}")


## 3. Stage 0 → 1 → 2 via pipeline classes

Primitives resolve calibrations from the KOA caldb using the ramp / L0 / L1 metadata.


In [ ]:
from hispecdrp.pipelines import Stage0Pipeline, Stage1Pipeline, Stage2Pipeline

# Stage 0 ramp-fit flags non-finite rate/rate_err as DO_NOT_USE; Stage 1 extraction
# also drops non-finite rate/err in the aperture mask. Keep the box on the profile core.
stage1_args = {"spec_extract": {"extraction_aperture": 5}}

l0 = Stage0Pipeline(output_dir=OUT).apply(ramp)

l1 = Stage1Pipeline(output_dir=OUT, prim_args=stage1_args).apply(l0)

# The template is stamped at 0 km/s topocentric (no barycentric shift), so disable the
# CCF barycentric term and center the search at 0. auto_center_scout off avoids spurious
# peak locks (the source of the old fiber-2 artifact).
rv_args = {
    "save_preview": True,
    "preview_write_png": False,
    "ccf_input_velocity": 0.0,
    "ccf_zb_from_barycorr": False,
    "mask_auto_center_scout": False,
}
l2 = Stage2Pipeline(output_dir=OUT, prim_args={"stellar_rv": rv_args}).apply(l1)

print(f"L0 type: {l0.meta.model_type}")
print(f"L0 filename: {l0.meta.filename}")
print(f"L1 type: {l1.meta.model_type}")
print(f"L1 filename: {l1.meta.filename}")
print(f"L2 type: {l2.meta.model_type}")
print(f"L2 filename: {l2.meta.filename}")


## 4. Stellar RV preview


In [ ]:
from IPython.display import HTML, display
from pathlib import Path
import html

preview = l2.get_output_path(output_dir=OUT, suffix="stellar_rv").replace(".fits", "_preview.html")
print(f"{preview} ({os.path.getsize(preview) / 1e6:.2f} MB)")
# iframe srcdoc so Plotly JS runs inside the notebook
display(HTML(
    f'<iframe srcdoc="{html.escape(Path(preview).read_text(encoding="utf-8"), quote=True)}" '
    f'width="100%" height="1400" style="border:0" sandbox="allow-scripts allow-same-origin"></iframe>'
))
